# 🦜 BirdCLEF 2026 - Training

> **Competition:** [BirdCLEF 2026](https://www.kaggle.com/competitions/birdclef-2026)  
> **Hardware:** Kaggle dual NVIDIA Tesla T4 × 2 (2 × 16 GB VRAM)  
> **Prerequisite:** Run the [I/O Preprocessing notebook](https://www.kaggle.com/code/emanuellcs/birdclef-2026-i-o-preprocessing) and the [Pseudo-Labeling notebook](https://www.kaggle.com/code/emanuellcs/birdclef-2026-target-domain-pseudo-labeling) first, and attach them as external datasets.

---

## Architecture Overview

This pipeline is engineered around a single objective: **sustain near-100 % GPU utilisation on
both T4 accelerators throughout training**. Every design decision below traces back to that goal,
from how audio reaches RAM all the way to the ONNX artefact written to disk.

### Three-Tier Precision Architecture

```text
╔══════════════════════════════════════════════════════════════════╗
║  TIER 0 - CPU RAM  (DataLoader Workers)                          ║
║  • 2 workers per DDP process, CPU-affinity-pinned to one core    ║
║  • Pre-pinned float32 tensors - zero-copy row-slice per sample   ║
║  • Pre-built one-hot target matrix - zero alloc in __getitem__   ║
╚═══════════════════════╦══════════════════════════════════════════╝
                        ║  PCIe DMA  (pin_memory + non_blocking=True)
╔═══════════════════════╩══════════════════════════════════════════╗
║  TIER 1 - GPU  AudioToMelPCEN  [forced float32]                  ║
║  • torch.stft()    -> cuFFT power spectrogram (B, F, T')         ║
║  • mel_fb matmul   -> (128 x 513) Mel filterbank projection      ║
║  • Depthwise conv  -> O(T) IIR background smoother via 1D Conv   ║
║    ^ t_idx buffer registered in __init__ - zero per-fwd alloc    ║
║  • PCEN formula    -> adaptive gain normalisation                ║
║    ^ @autocast('cuda', enabled=False) - eps=1e-6 safe in fp32    ║
╠══════════════════════════════════════════════════════════════════╣
║  TIER 2 - GPU  EfficientNet-B1 + GeM Pool + Masked Focal Loss    ║
║  • All Conv2d / BN / Linear ops in fp16 via torch.amp.autocast   ║
║  • cuDNN benchmark=True - autotuned kernels, fixed spectrogram   ║
╠══════════════════════════════════════════════════════════════════╣
║  DDP GRAD SYNC - NCCL all-reduce  (GPU-direct, zero GIL touch)   ║
║  • Gradients averaged via ring all-reduce before optimizer.step  ║
║  • running_loss += loss.detach()  -> ONE .item() sync per epoch  ║
╚══════════════════════════════════════════════════════════════════╝

```

### Why DDP over DataParallel?

`nn.DataParallel` scatters inputs and gathers outputs **through the CPU**, holding the Python GIL
for the entire gradient reduction. On a dual-T4 host, GPU:1 sits idle while GPU:0 reduces -
effectively halving throughput. `DistributedDataParallel` (DDP) with the NCCL backend performs
an **all-reduce directly between GPU memory buffers** (GPU-Direct RDMA over PCIe), bypassing
the CPU and the GIL entirely. Each GPU runs a fully independent Python process; synchronisation
happens only at the NCCL kernel boundary.

In [1]:
!pip install -q timm onnx onnxruntime onnxscript soundfile librosa tqdm psutil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 11.3 MB/s eta 0:00:00


## 0 - Configuration, Imports & Reproducibility

All hyperparameters live in a single `CFG` dictionary. This pattern has two advantages:

1. **Discoverability** - an engineer reading the notebook sees every tunable value in one
   place, without grepping for magic numbers scattered across function bodies.
2. **Propagation** - changing one value in `CFG` propagates to every function that
   receives it as an argument. There is no risk of updating a constant in one place
   and missing it in another.

Derived constants (`SAMPLES_PER_CLIP`, `N_GPUS`) are computed once from `CFG` immediately
below the dictionary, and are never re-derived inline throughout the notebook.

**Seeding strategy:** `seed_everything` covers Python's `random`, NumPy, and PyTorch
CPU RNGs. CUDA seeding (`torch.cuda.manual_seed_all`) is intentionally deferred to each
GPU worker process inside `train_worker` because CUDA device contexts do not exist in
the notebook process - they are created fresh in each spawned child process.

In [2]:
# -- Standard library ---------------------------------------------------------
import gc
import math
import ast
import os
import random
import time
import warnings
import sys
from copy import deepcopy
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# -- Third-party ---------------------------------------------------------------
import librosa          # CPU-only: used exclusively for Mel filterbank construction
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm

# -- PyTorch core --------------------------------------------------------------
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torch.utils.data.distributed import DistributedSampler

# -- Model zoo -----------------------------------------------------------------
import timm

# -- ONNX export stack ---------------------------------------------------------
import onnx
import onnxruntime as ort

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.dpi"] = 120

# =============================================================================
# GLOBAL CONFIGURATION
#
# Single source of truth for every hyperparameter in the pipeline.
# Pass `CFG` (or a subset) to every function that needs it - never read
# magic numbers from module scope or hardcode them inside function bodies.
# =============================================================================
CFG: Dict = dict(
    # -- Paths -----------------------------------------------------------------
    preprocessed_dir = Path(
        "/kaggle/input/notebooks/emanuellcs/birdclef-2026-i-o-preprocessing"
    ),
    output_dir = Path("/kaggle/working"),
    onnx_path  = "/kaggle/working/birdclef2026_fp32_opt.onnx",

    # -- Audio frontend --------------------------------------------------------
    # CRITICAL: these values must match the I/O Preprocessing notebook exactly.
    # Any mismatch silently corrupts the Mel projection at inference time.
    sample_rate   = 32_000,
    clip_duration = 5,            # seconds -> 160 000 samples
    n_fft         = 1024,
    hop_length    = 512,          # -> 313 time frames per 5-second clip
    n_mels        = 128,
    fmin          = 40,           # Hz - lowest audible bird frequency
    fmax          = 14_000,       # Hz - upper limit for T4 Nyquist (16 kHz)
    img_size      = (128, 313),   # (n_mels, n_time_frames)

    # -- Classification --------------------------------------------------------
    num_classes = 234,
    backbone    = 'tf_efficientnet_b1',

    # -- Training schedule -----------------------------------------------------
    batch_size = 64,   # global batch; each GPU receives batch_size // world_size

    # mp.start_processes forks two independent processes; each spawns 2 DataLoader
    # workers -> 4 total workers, one per physical core, zero context-switch overhead.
    num_workers      = 2,
    epochs           = 30,
    train_fold       = 0,
    n_folds          = 5,
    lr               = 3e-4,
    weight_decay     = 1e-4,
    # Layer-wise Learning Rate Decay (LLRD): backbone LR = lr * backbone_lr_mult.
    # The pretrained backbone fine-tunes gently while the head trains at full speed.
    backbone_lr_mult = 0.1,
    snapshot_epochs  = [20, 22, 24, 26, 28, 30],

    # -- Additive MixUp --------------------------------------------------------
    mixup_prob      = 0.50,
    mixup_alpha_min = 0.3,
    mixup_alpha_max = 0.7,

    # -- Waveform augmentations ------------------------------------------------
    bg_mix_prob     = 0.80,
    bg_snr_db_min   = 10.0,
    bg_snr_db_max   = 30.0,
    time_shift_prob = 0.50,
    gain_prob       = 0.40,
    gain_db_min     = -12.0, # Asymmetric range to prevent clipping
    gain_db_max     = 2.0,

    # -- SpecAugment -----------------------------------------------------------
    freq_mask_param = 12,
    time_mask_param = 24,
    n_freq_masks    = 2,
    n_time_masks    = 2,

    # -- Noisy Student / Pseudo-Labeling ---------------------------------------
    pseudo_csv_path     = Path("/kaggle/input/notebooks/emanuellcs/birdclef-2026-target-domain-pseudo-labeling/pseudo_labels_gamma1.csv"),
    use_pseudo_labels   = True,
    pseudo_label_weight = 0.5,
)

# -- Derived constants (computed once; never re-derived inline) -----------------
SAMPLES_PER_CLIP: int = CFG["sample_rate"] * CFG["clip_duration"]  # 160_000
DEVICE: torch.device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPUS: int           = torch.cuda.device_count()
CFG["output_dir"].mkdir(parents=True, exist_ok=True)

print(f"[ENV]  Device   : {DEVICE}")
print(f"[ENV]  GPUs     : {N_GPUS}")
print(f"[ENV]  Clip len : {SAMPLES_PER_CLIP:,} samples  "
      f"({CFG['clip_duration']}s @ {CFG['sample_rate']} Hz)")


def seed_everything(seed: int = 42) -> None:
    """Seed all non-CUDA RNGs for reproducible data-loading and augmentation.

    CUDA seeding is deferred to each GPU worker process inside ``train_worker``
    because CUDA device contexts do not exist in the notebook process at this
    point - they are created fresh inside each spawned child process.

    Args:
        seed: Integer seed value applied to ``random``, ``numpy``, and the
            PyTorch CPU RNG. Also written to ``PYTHONHASHSEED`` to stabilise
            Python's built-in hash randomisation. Defaults to 42.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


seed_everything(42)
print("[SEED] All CPU RNGs seeded to 42.")

[ENV]  Device   : cuda
[ENV]  GPUs     : 2
[ENV]  Clip len : 160,000 samples  (5s @ 32000 Hz)
[SEED] All CPU RNGs seeded to 42.


## 1 - `worker.py`: The DDP Training Module

### Why does `worker.py` need to be a separate file?

PyTorch's `mp.start_processes` with `start_method='spawn'` launches a **fresh Python
interpreter** for each GPU process. That interpreter must import the entry-point function
(`train_worker`) by name from a proper Python module. Code defined interactively in a
Jupyter cell lives in a frozen `__main__` namespace that spawned processes cannot import,
raising:

```
AttributeError: Can't get attribute 'train_worker' on
<module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>
```

The `%%writefile` magic writes the entire module to disk **before any spawning occurs**,
making it importable via `from worker import train_worker`.

> **Why not `fork`?** NCCL explicitly documents that `fork` is unsafe when a CUDA context
> has already been initialised in the parent process. The `spawn` start method is the only
> option that guarantees a clean CUDA context in each child.

### Module Organisation

```text
worker.py
  SECTION 1  AudioToMelPCEN       GPU acoustic frontend (STFT -> Mel -> PCEN)
  SECTION 2  RAMBirdDataset        Zero-copy memory-mapped waveform dataset
             worker_init_fn        Per-DataLoader-worker CPU-affinity setup
             build_spec_augment    SpecAugment frequency + time masking
  SECTION 3  MaskedBCEWithLogitsLoss    Masked Focal Loss for noisy secondary labels
             GeM                   Generalized Mean Pooling for localized bird chirps
             BirdCLEFModel         End-to-end model (frontend + backbone + head)
             build_optimizer       AdamW with Layer-wise Learning Rate Decay
  SECTION 4  build_dataloaders_ddp DDP-aware DataLoader factory
  SECTION 5  batch_waveform_augment Vectorised GPU waveform augmentations (Soft Label Blending)
             train_one_epoch       Single-epoch training loop
             validate              Validation loop (loss + macro ROC-AUC)
  SECTION 6  ddp_setup             NCCL process-group initialisation
             ddp_cleanup           Process-group teardown
  SECTION 7  train_worker          Per-GPU entry point (mp.start_processes target)

```

In [3]:
%%writefile /kaggle/working/worker.py
# =============================================================================
# worker.py -- BirdCLEF 2026 DDP Training Module
#
# Written to disk by the notebook cell above so that mp.start_processes
# (spawn method) can import train_worker by name from a proper module.
# All symbols that train_worker depends on must live in this file.
#
# Usage from the notebook:
#   import sys; sys.path.insert(0, '/kaggle/working')
#   from worker import train_worker
#   mp.start_processes(train_worker, args=(...), nprocs=world_size,
#                      join=True, start_method='spawn')
# =============================================================================

# -- Standard library ---------------------------------------------------------
import gc
import math
import os
import random
import time
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# -- Third-party --------------------------------------------------------------
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

# -- PyTorch ------------------------------------------------------------------
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.distributed import DistributedSampler

import timm

warnings.filterwarnings('ignore')


# =============================================================================
# SECTION 1 -- GPU ACOUSTIC FRONTEND
# =============================================================================

class AudioToMelPCEN(nn.Module):
    """GPU-native acoustic frontend: float32 waveform -> PCEN Mel-spectrogram.

    Pipeline
    --------
    waveform (B, T)
        -> torch.stft()          cuFFT power spectrogram  (B, F, T')
        -> mel_fb @ power        Mel filterbank projection (B, M, T')
        -> _fft_causal_smooth()  IIR background smoother via 1D Conv (B, M, T')
        -> PCEN formula          adaptive gain normalisation (B, M, T')
        -> unsqueeze(1)          channel dim for backbone  (B, 1, M, T')

    Precision Contract
    ------------------
    ``@torch.amp.autocast('cuda', enabled=False)`` forces the entire forward
    method to run in float32, even when called inside an outer autocast context.
    This is mandatory: PCEN uses eps=1e-6 as a denominator guard, but float16's
    smallest normal value is ~6e-5 -- PCEN would silently underflow in fp16.

    Depthwise Causal 1D Convolution (IIR Smoother)
    ----------------------------------------------
    The PCEN background estimate M is defined by the recurrence:
        M[t] = (1 - s) * M[t-1] + s * S[t],  M[0] = S[0]

    A naive sequential loop is O(T) and not GPU-parallelisable. We reformulate
    it as an exact mathematical equivalent: a depthwise causal 1D convolution.
    Since T=313 is small, this is extremely fast on GPUs and perfectly stable 
    in ONNX, bypassing the limitations of PyTorch's `irfft` in ONNX exports.

    Args:
        cfg: Global configuration dict. Required keys: ``sample_rate``,
            ``n_fft``, ``hop_length``, ``n_mels``, ``fmin``, ``fmax``,
            ``img_size``.
    """

    def __init__(self, cfg: Dict) -> None:
        super().__init__()

        sr    = cfg['sample_rate']
        n_fft = cfg['n_fft']
        hop   = cfg['hop_length']
        n_mels, fmin, fmax = cfg['n_mels'], cfg['fmin'], cfg['fmax']

        self.n_fft = n_fft
        self.hop   = hop
        self.eps   = 1e-6  # PCEN denominator guard -- DO NOT reduce below 1e-6

        # -- Mel filterbank (128 x 513, slaney-normalised, HTK=False) ----------
        # Built once on CPU by librosa; registered as a GPU buffer so it is
        # moved automatically by .to(device) and preserved across checkpoints.
        mel_fb = librosa.filters.mel(
            sr=sr, n_fft=n_fft, n_mels=n_mels,
            fmin=fmin, fmax=fmax, norm='slaney', htk=False,
        ).astype(np.float32)
        self.register_buffer('mel_fb', torch.from_numpy(mel_fb))

        # -- Hann window -------------------------------------------------------
        # Registered as a buffer (not a parameter) so torch.stft() receives a
        # tensor on the same device as the waveform without extra transfers.
        self.register_buffer('hann_window', torch.hann_window(n_fft))

        # -- Pre-allocated causal-kernel time index ----------------
        # t_idx[t] = t, used to compute K[t] = (1-s)^t in _fft_causal_smooth.
        # Registering it as a buffer avoids re-allocation on every forward pass.
        T_frames = cfg['img_size'][1]  # STFT time frames (313 for 5s clips)
        self.register_buffer('t_idx', torch.arange(T_frames, dtype=torch.float32))

        # -- Learnable PCEN parameters (shape: 1 x n_mels x 1) ----------------
        # Stored in unconstrained space; constrained via sigmoid/softplus in
        # forward() to enforce alpha in (0,1), delta > 0, r in (0,1), s in (0,0.1).
        # The singleton dimensions enable free broadcasting over (B, M, T').
        def _logit(x: float) -> float:
            """Inverse sigmoid: maps x in (0,1) to the real line."""
            return math.log(x / (1.0 - x))

        def _softplus_inv(x: float) -> float:
            """Inverse softplus: maps x > 0 to the real line."""
            return math.log(math.exp(x) - 1.0)

        shape = (1, n_mels, 1)
        self.alpha_raw = nn.Parameter(torch.full(shape, _logit(0.98)))
        self.delta_raw = nn.Parameter(torch.full(shape, _softplus_inv(2.0)))
        self.r_raw     = nn.Parameter(torch.full(shape, _logit(0.5)))
        self.s_raw     = nn.Parameter(torch.full(shape, _logit(0.25)))

    def _fft_causal_smooth(
        self,
        S: torch.Tensor,  # (B, M, T) -- Mel power, clamped to [eps, inf)
        s: torch.Tensor,  # (1, M, 1) -- smoothing rate in (0, 0.1)
    ) -> torch.Tensor:
        """Compute the IIR background smoother M via 1D Convolution.

        PyTorch's irfft translates poorly to ONNX because the ONNX DFT spec 
        forbids 'is_onesided' and 'inverse' from being enabled simultaneously.
        
        We replace the FFT with an exact mathematical equivalent: a depthwise 
        causal 1D convolution. Since T=313 is small, this is extremely fast 
        on GPUs and perfectly stable in ONNX.
        """
        B, M_bands, T = S.shape

        a = 1.0 - s  # forget factor, shape (1, M, 1)

        # Build X_tilde: initial condition is S[0]; subsequent frames scaled by s.
        X_init  = S[..., :1]                           # (B, M, 1)
        X_rest  = s * S[..., 1:]                       # (B, M, T-1)
        X_tilde = torch.cat([X_init, X_rest], dim=-1)  # (B, M, T)

        # Exponential decay kernel K[t] = a^t
        K = a.pow(self.t_idx[:T])                      # (1, M, T)
        
        # PyTorch conv1d computes cross-correlation: sum_k X[t+k] W[k]
        # To make it a true causal convolution, we flip the kernel in time.
        W = K.flip(dims=[-1])                          # (1, M, T)
        
        # Reshape for depthwise conv1d: (out_channels, in_channels/groups, kernel_size)
        # Groups = M_bands, so the weight must be shaped (M, 1, T)
        weight = W.transpose(0, 1)                     # (M, 1, T)

        # Left-pad X_tilde by T-1 to ensure the convolution only looks into the past
        X_padded = F.pad(X_tilde, (T - 1, 0))          # (B, M, 2T - 1)

        # Execute the depthwise causal convolution
        M_hat = F.conv1d(X_padded, weight, groups=M_bands)  # (B, M, T)

        return M_hat

    @torch.amp.autocast('cuda', enabled=False)
    def forward(self, wav: torch.Tensor) -> torch.Tensor:
        """Transform a batch of waveforms into PCEN Mel-spectrograms.

        The ``@autocast('cuda', enabled=False)`` decorator enforces float32
        execution even when called inside an outer autocast context block.
        See class docstring for the precision contract rationale.

        Args:
            wav: Raw waveform tensor of shape ``(B, T)``, float32, in [-1, 1].

        Returns:
            PCEN Mel-spectrogram of shape ``(B, 1, n_mels, T')``, float32.
            The channel dimension (1) is prepended so the tensor can be passed
            directly to any single-channel image backbone (in_chans=1).
        """
        wav = wav.float()  # ensure float32 under any outer autocast context

        # -- STFT -> power spectrogram ----------------------------------------
        stft  = torch.stft(
            wav, n_fft=self.n_fft, hop_length=self.hop,
            win_length=self.n_fft, window=self.hann_window,
            return_complex=True, normalized=False,
        )                                     # (B, F, T')
        power = stft.abs().pow(2)             # (B, F, T')

        # -- Mel filterbank projection ----------------------------------------
        S = torch.matmul(self.mel_fb, power)  # (B, M, T')
        S = S.clamp(min=self.eps)             # guard log(0) in PCEN denominator

        # -- Decode PCEN parameters from unconstrained space ------------------
        alpha = torch.sigmoid(self.alpha_raw)      # (1, M, 1) in (0, 1)
        delta = F.softplus(self.delta_raw)         # (1, M, 1) > 0
        r     = torch.sigmoid(self.r_raw)          # (1, M, 1) in (0, 1)
        s     = torch.sigmoid(self.s_raw) * 0.1   # (1, M, 1) in (0, 0.1)

        # -- FFT-based IIR background smoother --------------------------------
        M_hat = self._fft_causal_smooth(S, s)      # (B, M, T')

        # Catch FFT numerical ringing to prevent NaN in fractional power
        M_hat = torch.clamp(M_hat, min=0.0)

        # -- PCEN: (S / (eps + M)^alpha + delta)^r - delta^r -----------------
        denom = (self.eps + M_hat).pow(alpha)
        pcen  = (S / denom + delta).pow(r) - delta.pow(r)  # (B, M, T')

        return pcen.unsqueeze(1)  # (B, 1, M, T')


# =============================================================================
# SECTION 2 -- MEMORY-MAPPED DATASET & WORKER INITIALISATION
# =============================================================================

class RAMBirdDataset(Dataset):
    """Zero-copy memory-mapped waveform dataset supporting Soft Pseudo-Labels."""

    def __init__(
        self,
        meta_df:        pd.DataFrame,
        focal_npy_path: str,
        bg_npy_path:    str,
        cfg:            Dict,
        is_train:       bool = True,
    ) -> None:
        self.is_train       = is_train
        self.focal_npy_path = focal_npy_path
        self.bg_npy_path    = bg_npy_path
        self.mmap_indices   = meta_df['mmap_index'].values.astype(np.int64)
        self.label_ids      = meta_df['label_id'].values.astype(np.int64)

        self._waveforms: Optional[np.ndarray] = None
        self._bg_waves:  Optional[np.ndarray] = None

        n = len(self.mmap_indices)
        
        # 1. Build Primary Targets
        self.primary_targets = torch.zeros(n, cfg['num_classes'], dtype=torch.float32)
        self.primary_targets[torch.arange(n), torch.from_numpy(self.label_ids)] = 1.0
        
        # 2. Build Secondary Targets
        self.secondary_targets = torch.zeros(n, cfg['num_classes'], dtype=torch.float32)
        lbl2id = cfg.get('label_map', {})
        
        if 'secondary_labels' in meta_df.columns:
            for i, sec_str in enumerate(meta_df['secondary_labels'].fillna("[]")):
                try:
                    sec_list = ast.literal_eval(sec_str)
                    for species in sec_list:
                        if species in lbl2id:
                            self.secondary_targets[i, lbl2id[species]] = 1.0
                except Exception:
                    pass

        # 3. Load Soft Pseudo-Labels for Background (STAGE 2 NOISY STUDENT)
        self.use_pseudo = cfg.get('use_pseudo_labels', False) and is_train
        if self.use_pseudo and Path(cfg['pseudo_csv_path']).exists():
            # Assuming the CSV rows map 1:1 to the sliding windows in bg_npy_path
            pseudo_df = pd.read_csv(cfg['pseudo_csv_path'])
            
            # Extract only the class probability columns (ignore filename, start, end)
            class_cols = [c for c in pseudo_df.columns if c in lbl2id.keys()]
            
            # Reorder columns to exactly match the network's class indices
            ordered_cols = sorted(class_cols, key=lambda x: lbl2id[x])
            
            self.bg_targets = torch.from_numpy(pseudo_df[ordered_cols].values.astype(np.float32))
            print(f"[DATA] Loaded {len(self.bg_targets)} pseudo-labeled background targets.")
        else:
            self.bg_targets = None

    def _init_arrays(self) -> None:
        if self._waveforms is None:
            self._waveforms = np.load(self.focal_npy_path, mmap_mode='r')
            self._bg_waves  = (
                self._waveforms if self.focal_npy_path == self.bg_npy_path
                else np.load(self.bg_npy_path, mmap_mode='r')
            )

    def __len__(self) -> int:
        return len(self.mmap_indices)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, ...]:
        self._init_arrays()

        mmap_idx = int(self.mmap_indices[index])
        wav      = torch.from_numpy(self._waveforms[mmap_idx].copy())
        
        p_target = self.primary_targets[index]
        s_target = self.secondary_targets[index]

        if self.is_train:
            bg_idx = random.randint(0, len(self._bg_waves) - 1)
            bg     = torch.from_numpy(self._bg_waves[bg_idx].copy())
            
            # Retrieve the soft label for this specific background chunk, or zeros if disabled
            bg_tgt = self.bg_targets[bg_idx] if self.bg_targets is not None else torch.zeros_like(p_target)
            return wav, bg, p_target, s_target, bg_tgt

        return wav, p_target, s_target


def worker_init_fn(worker_id: int) -> None:
    """Per-DataLoader-worker initialisation called at worker process spawn time.

    Args:
        worker_id: Zero-based worker index assigned by DataLoader at spawn.
    """
    try:
        import psutil
        psutil.Process().cpu_affinity([worker_id % os.cpu_count()])
    except Exception:
        pass
    # Re-seed Python's RNG per worker so augmentations across workers are
    # not correlated (each worker derives its seed from its initial_seed).
    random.seed(torch.initial_seed() % (2 ** 32))


def build_spec_augment(cfg: Dict) -> nn.Sequential:
    """Build a SpecAugment pipeline for on-GPU frequency and time masking.

    SpecAugment (Park et al., 2019) applies random rectangular masks to the
    spectrogram along the frequency and time axes. Frequency masking prevents
    the model from relying on any single frequency band; time masking prevents
    over-fitting to specific temporal positions in the clip.

    The pipeline executes **on the GPU after PCEN** (inside BirdCLEFModel.forward),
    so it contributes zero CPU time to the DataLoader hot path.

    Args:
        cfg: Configuration dict. Required keys: ``n_freq_masks``,
            ``freq_mask_param``, ``n_time_masks``, ``time_mask_param``.

    Returns:
        An ``nn.Sequential`` of ``FrequencyMasking`` and ``TimeMasking``
        transforms from ``torchaudio.transforms``.
    """
    import torchaudio.transforms as T

    layers: List[nn.Module] = []
    for _ in range(cfg['n_freq_masks']):
        layers.append(T.FrequencyMasking(freq_mask_param=cfg['freq_mask_param']))
    for _ in range(cfg['n_time_masks']):
        layers.append(T.TimeMasking(time_mask_param=cfg['time_mask_param']))

    return nn.Sequential(*layers)


# =============================================================================
# SECTION 3 -- MODEL, ASYMMETRIC LOSS & OPTIMISER
# =============================================================================

class MaskedBCEWithLogitsLoss(nn.Module):
    """
    Implements Loss Masking for noisy secondary targets.
    """
    def __init__(self) -> None:
        super().__init__()
        self.criterion = nn.BCEWithLogitsLoss(reduction='none')

    def forward(
        self, 
        logits: torch.Tensor, 
        primary_targets: torch.Tensor, 
        secondary_targets: torch.Tensor
    ) -> torch.Tensor:
        
        combined_targets = torch.clamp(primary_targets + secondary_targets, 0.0, 1.0)
        raw_loss = self.criterion(logits, combined_targets)
        
        mask = torch.ones_like(raw_loss)
        mask[secondary_targets == 1.0] = 0.0
        mask[primary_targets == 1.0] = 1.0
        
        return (raw_loss * mask).mean()

class SEDAttentionHead(nn.Module):
    """
    Temporal attention mechanism to focus on specific time frames containing 
    vocalizations while ignoring background noise frames.
    """
    def __init__(self, in_features: int, num_classes: int):
        super().__init__()
        self.fc = nn.Linear(in_features, num_classes)
        self.attention = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.Tanh(),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (B, Channels, Freq, Time)
        x = torch.mean(x, dim=2)  # Pool frequency (B, C, Time)
        x = x.transpose(1, 2)     # (B, Time, C)
        
        framewise_logits = self.fc(x)  # (B, Time, num_classes)
        att_weights = self.attention(x)
        att_weights = torch.softmax(att_weights, dim=1) 
        
        clipwise_logits = torch.sum(framewise_logits * att_weights, dim=1) 
        return clipwise_logits

class BirdCLEFModel(nn.Module):
    """End-to-end BirdCLEF model with SED Head."""
    def __init__(self, cfg: Dict, is_train: bool = True) -> None:
        super().__init__()
        self.is_train = is_train

        self.frontend = AudioToMelPCEN(cfg)
        self.spec_aug = build_spec_augment(cfg)
        
        # Load backbone without Global Average Pooling (global_pool='')
        self.backbone = timm.create_model(
            cfg['backbone'], pretrained=True,
            in_chans=1, num_classes=0, global_pool='',
        )
        
        self.head = SEDAttentionHead(
            in_features=self.backbone.num_features, 
            num_classes=cfg['num_classes']
        )

    def forward(self, wav: torch.Tensor) -> torch.Tensor:
        spec = self.frontend(wav)
        if self.is_train and self.training:
            spec = self.spec_aug(spec)
        x = self.backbone(spec)
        return self.head(x)

def build_optimizer(model: nn.Module, cfg: Dict) -> AdamW:
    """Construct an AdamW optimiser with Layer-wise Learning Rate Decay."""
    return AdamW(
        [
            {'params': list(model.frontend.parameters()),
             'lr': cfg['lr'],
             'name': 'pcen_frontend'},
            {'params': list(model.head.parameters()),
             'lr': cfg['lr'],
             'name': 'classifier_head'},
            {'params': list(model.backbone.parameters()),
             'lr': cfg['lr'] * cfg['backbone_lr_mult'],
             'name': 'backbone'},
        ],
        weight_decay=cfg['weight_decay'],
    )


# =============================================================================
# SECTION 4 -- DDP-COMPATIBLE DATALOADER FACTORY
# =============================================================================

def build_dataloaders_ddp(
    meta_df:        pd.DataFrame,
    focal_npy_path: str,
    bg_npy_path:    str,
    cfg:            Dict,
    fold:           int,
    rank:           int,
    world_size:     int,
) -> Tuple[DataLoader, DataLoader, DistributedSampler]:
    """Build train and validation DataLoaders for one DDP process.

    Each GPU process calls this independently with its own ``rank``.
    ``DistributedSampler`` partitions the dataset so no two ranks see the
    same sample in a given epoch. ``drop_last=True`` on the train sampler
    ensures all ranks process an identical number of batches, which is required
    for ``all_reduce`` to not deadlock (all ranks must call it the same number
    of times).

    Waveform array paths are passed as strings (not numpy arrays) so that
    DataLoader workers can open their own mmap handles independently, without
    pickling potentially large arrays across the process boundary.

    Args:
        meta_df: Full metadata DataFrame (un-split; the split happens here).
        focal_npy_path: Path to the focal waveform array (.npy).
        bg_npy_path: Path to the background waveform array (.npy).
        cfg: Configuration dict. Required keys: ``n_folds``, ``batch_size``,
            ``num_workers``.
        fold: Fold index to use for the train/validation split.
        rank: DDP rank of the calling process.
        world_size: Total number of DDP processes.

    Returns:
        A 3-tuple ``(train_loader, val_loader, train_sampler)``.
        ``train_sampler`` is returned so the caller can call
        ``train_sampler.set_epoch(epoch)`` for correct per-epoch reshuffling.
    """
    # -- Stratified K-fold split ----------------------------------------------
    skf    = StratifiedKFold(n_splits=cfg['n_folds'], shuffle=True, random_state=42)
    splits = list(skf.split(meta_df, meta_df['label_id']))
    train_idx, val_idx = splits[fold]

    df_train = meta_df.iloc[train_idx].copy()
    df_val   = meta_df.iloc[val_idx].copy()

    if rank == 0:
        print(f'[FOLD {fold}] Train: {len(df_train):,}  |  Val: {len(df_val):,}')

    # -- Dataset construction -------------------------------------------------
    train_ds = RAMBirdDataset(df_train, focal_npy_path, bg_npy_path, cfg, is_train=True)
    val_ds   = RAMBirdDataset(df_val,   focal_npy_path, bg_npy_path, cfg, is_train=False)

    # -- Distributed samplers -------------------------------------------------
    # drop_last=True guarantees equal batch counts per rank for all_reduce.
    train_sampler = DistributedSampler(
        train_ds, num_replicas=world_size, rank=rank,
        shuffle=True, drop_last=True,
    )
    val_sampler = DistributedSampler(
        val_ds, num_replicas=world_size, rank=rank,
        shuffle=False, drop_last=False,
    )

    # -- DataLoader construction ----------------------------------------------
    # pin_memory=True: allocates page-locked CPU buffers so DMA to GPU can
    #   bypass an intermediate copy, reducing PCIe transfer time.
    # persistent_workers=True: keeps worker processes alive between epochs to
    #   amortise the ~200 ms spawn overhead that would otherwise repeat each epoch.
    # prefetch_factor=2: each worker pre-fetches 2 batches ahead of the GPU
    #   to hide DataLoader latency behind GPU compute.
    num_workers     = cfg['num_workers']
    prefetch_factor = 2

    train_loader = DataLoader(
        train_ds,
        batch_size         = cfg['batch_size'] // world_size,
        sampler            = train_sampler,
        num_workers        = num_workers,
        pin_memory         = True,
        persistent_workers = True,
        prefetch_factor    = prefetch_factor,
        drop_last          = True,
        worker_init_fn     = worker_init_fn,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size         = (cfg['batch_size'] * 2) // world_size,
        sampler            = val_sampler,
        num_workers        = num_workers,
        pin_memory         = True,
        persistent_workers = True,
        prefetch_factor    = prefetch_factor,
        worker_init_fn     = worker_init_fn,
    )

    return train_loader, val_loader, train_sampler


# =============================================================================
# SECTION 5 -- TRAINING & VALIDATION LOOPS
# =============================================================================

def batch_waveform_augment(
    wavs: torch.Tensor,
    bgs:  torch.Tensor,
    p_targets: torch.Tensor,  # Pass the primary targets in
    bg_targets: torch.Tensor, # Pass the soft background targets in
    cfg:  Dict,
) -> Tuple[torch.Tensor, torch.Tensor]:
    B, T   = wavs.shape
    device = wavs.device

    # 1. Background Mix with SOFT LABEL BLENDING
    mix_mask  = (torch.rand(B, 1, device=device) < cfg['bg_mix_prob']).float()
    rms_focal = torch.sqrt(torch.mean(wavs ** 2, dim=1, keepdim=True) + 1e-9)
    rms_bg    = torch.sqrt(torch.mean(bgs  ** 2, dim=1, keepdim=True) + 1e-9)
    snr_db    = torch.empty(B, 1, device=device).uniform_(cfg['bg_snr_db_min'], cfg['bg_snr_db_max'])
    
    scale = rms_focal / (rms_bg * (10.0 ** (snr_db / 20.0)))
    wavs  = torch.clamp(wavs + bgs * scale * mix_mask, -1.0, 1.0)

    # BLEND TARGETS: Scale the pseudo-labels by the mixing volume and hyperparameter weight
    if cfg.get('use_pseudo_labels', False):
        blended_targets = p_targets + (bg_targets * scale * mix_mask * cfg['pseudo_label_weight'])
        p_targets = torch.clamp(blended_targets, 0.0, 1.0)

    # 2. Time Shift
    shift_mask = (torch.rand(B, 1, device=device) < cfg['time_shift_prob']).long()
    shifts     = torch.randint(0, T, (B, 1), device=device) * shift_mask
    base_idx   = torch.arange(T, device=device).unsqueeze(0).expand(B, T)
    gather_idx = (base_idx - shifts) % T
    wavs       = torch.gather(wavs, dim=1, index=gather_idx)

    # 3. Random Gain
    gain_mask   = (torch.rand(B, 1, device=device) < cfg['gain_prob']).float()
    gain_db     = torch.empty(B, 1, device=device).uniform_(cfg['gain_db_min'], cfg['gain_db_max'])
    linear_gain = 1.0 + (10.0 ** (gain_db / 20.0) - 1.0) * gain_mask
    wavs        = torch.clamp(wavs * linear_gain, -1.0, 1.0)

    return wavs, p_targets


def train_one_epoch(
    model:     nn.Module,
    loader:    DataLoader,
    optimizer: AdamW,
    criterion: MaskedBCEWithLogitsLoss,
    scaler:    torch.amp.GradScaler,
    device:    torch.device,
    epoch:     int,
    cfg:       Dict,
) -> float:
    """Execute one full training epoch with mixed-precision and DDP."""
    model.train()
    running_loss = torch.tensor(0.0, device=device)
    progress = tqdm(loader, desc=f'Epoch {epoch:03d} [train]', leave=False)

    for wav_batch, bg_batch, primary_targets, secondary_targets, bg_targets in progress:
        wav_batch = wav_batch.to(device, non_blocking=True)
        bg_batch  = bg_batch.to(device,  non_blocking=True)
        primary_targets   = primary_targets.to(device, non_blocking=True)
        secondary_targets = secondary_targets.to(device, non_blocking=True)
        bg_targets        = bg_targets.to(device, non_blocking=True)

        # Overwrite wav_batch AND primary_targets with the blended outputs
        wav_batch, primary_targets = batch_waveform_augment(
            wav_batch, bg_batch, primary_targets, bg_targets, cfg
        )

        # 2. ADDITIVE MIXUP (Focal + Focal)
        if torch.rand(1).item() < cfg['mixup_prob']:
            B = wav_batch.shape[0]
            indices = torch.randperm(B, device=device)
            wav_shuffled = wav_batch[indices]
            
            p_targets_shuffled = primary_targets[indices]
            s_targets_shuffled = secondary_targets[indices]

            alpha = torch.empty(B, 1, device=device).uniform_(cfg['mixup_alpha_min'], cfg['mixup_alpha_max'])
            beta  = torch.empty(B, 1, device=device).uniform_(cfg['mixup_alpha_min'], cfg['mixup_alpha_max'])

            wav_batch = torch.clamp((alpha * wav_batch) + (beta * wav_shuffled), -1.0, 1.0)
            primary_targets = torch.max(primary_targets, p_targets_shuffled)
            secondary_targets = torch.max(secondary_targets, s_targets_shuffled)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            logits = model(wav_batch)
            loss   = criterion(logits, primary_targets, secondary_targets)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.detach()

        if progress.n % 20 == 0:
            progress.set_postfix({'scale': f'{scaler.get_scale():.0f}'})

    return (running_loss / len(loader)).item()


@torch.no_grad()
def validate(
    model:     nn.Module,
    loader:    DataLoader,
    criterion: MaskedBCEWithLogitsLoss,
    device:    torch.device,
) -> Tuple[float, float]:
    """Run a full validation pass and compute mean loss and macro ROC-AUC."""
    model.eval()

    all_logits:  List[torch.Tensor] = []
    all_targets: List[torch.Tensor] = []
    running_loss = torch.tensor(0.0, device=device)

    for wav_batch, primary_targets, secondary_targets in tqdm(loader, desc='[val]', leave=False):
        wav_batch = wav_batch.to(device, non_blocking=True)
        primary_targets   = primary_targets.to(device, non_blocking=True)
        secondary_targets = secondary_targets.to(device, non_blocking=True)

        with torch.amp.autocast('cuda'):
            logits = model(wav_batch)
            running_loss += criterion(logits, primary_targets, secondary_targets).detach()

        all_logits.append(logits.detach())
        # We only evaluate ROC-AUC against the verified primary targets
        all_targets.append(primary_targets.detach())

    stacked_logits  = torch.cat(all_logits,  dim=0).float().cpu().numpy()
    stacked_targets = torch.cat(all_targets, dim=0).float().cpu().numpy()

    probs = 1.0 / (1.0 + np.exp(-stacked_logits))

    try:
        per_class_aucs = roc_auc_score(stacked_targets, probs, average=None)
        auc = float(np.nanmean(per_class_aucs))
        if np.isnan(auc):
            auc = 0.0
    except ValueError:
        auc = 0.0

    mean_loss = (running_loss / len(loader)).item()
    return mean_loss, auc


# =============================================================================
# SECTION 6 -- DDP PROCESS-GROUP LIFECYCLE
# =============================================================================

def ddp_setup(rank: int, world_size: int) -> None:
    """Initialise the NCCL process group for a single rank.

    All ``world_size`` processes must call this before any DDP communication
    primitive (``all_reduce``, etc.) can be used. The call blocks until all
    ranks have joined the rendezvous.

    The ``NCCL_P2P_DISABLE`` and ``NCCL_IB_DISABLE`` flags suppress spurious
    NCCL warnings on Kaggle T4 instances, which have neither NVLink (for P2P
    GPU memory transfers) nor InfiniBand (for RDMA fabric).

    Args:
        rank: Zero-based GPU rank for this process.
        world_size: Total number of participating processes.
    """
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    # Disable P2P and IB to avoid NCCL warnings on T4 hardware.
    os.environ.setdefault('NCCL_P2P_DISABLE', '1')
    os.environ.setdefault('NCCL_IB_DISABLE',  '1')

    dist.init_process_group(backend='nccl', rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)


def ddp_cleanup() -> None:
    """Destroy the NCCL process group and release all distributed resources.

    Must be called at the end of every ``train_worker`` invocation, including
    error paths. Failure to call this leaves the rendezvous store open,
    causing the next ``mp.start_processes`` call to hang at initialisation.
    """
    dist.destroy_process_group()


# =============================================================================
# SECTION 7 -- TRAINING WORKER ENTRY POINT
# =============================================================================

def train_worker(
    rank:       int,
    world_size: int,
    cfg:        Dict,
    meta_df:    pd.DataFrame,
    focal_npy:  str,  # path string -- loaded inside worker, not a pickled array
    bg_npy:     str,  # path string -- loaded inside worker, not a pickled array
) -> None:
    """Per-GPU training process, invoked once per GPU by ``mp.start_processes``.

    Responsibilities
    ----------------
    - Initialise the NCCL process group (``ddp_setup``).
    - Build the DDP-wrapped model, optimiser, scheduler, and loss function.
    - Run the training loop for ``cfg['epochs']`` epochs.
    - Save snapshot checkpoints and the best-AUC model to disk (rank 0 only).
    - Aggregate per-rank metrics via ``all_reduce`` so rank 0 logs the true
      global average (not just the rank-0 data shard's value).
    - Destroy the process group on exit (``ddp_cleanup``).

    Checkpoint Convention
    ---------------------
    Checkpoints are saved as ``model.module.state_dict()`` -- the state dict
    of the unwrapped inner model, without the DDP wrapper. This makes them
    directly loadable by any standard ``nn.Module`` without requiring DDP at
    inference time.

    Array Loading
    -------------
    ``focal_npy`` and ``bg_npy`` are path strings, not numpy arrays. Loading
    inside the worker (rather than in the notebook and passing the array)
    avoids pickling potentially large numpy arrays across the process boundary
    (numpy mmap arrays are not picklable; in-memory arrays can be hundreds of MB).

    Args:
        rank: Zero-based GPU rank assigned by ``mp.start_processes``.
            Rank 0 handles all logging and checkpointing.
        world_size: Total number of GPU processes.
        cfg: Global configuration dictionary.
        meta_df: Full metadata DataFrame (split inside ``build_dataloaders_ddp``).
        focal_npy: Filesystem path to the focal waveform .npy array.
        bg_npy: Filesystem path to the background waveform .npy array.
    """
    ddp_setup(rank, world_size)

    # benchmark=True enables cuDNN to autotune its convolution
    # kernel selection for the fixed spectrogram shape (128 x 313). This is
    # a one-time cost at the start of training that yields ~10-15% throughput
    # improvement for fixed-size inputs. deterministic=False is required because
    # cuDNN's fastest kernels use non-deterministic parallel reductions.
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.benchmark     = True
    torch.backends.cudnn.deterministic = False

    device = torch.device(f'cuda:{rank}')

    train_loader, val_loader, train_sampler = build_dataloaders_ddp(
        meta_df, focal_npy, bg_npy,
        cfg=cfg, fold=cfg['train_fold'],
        rank=rank, world_size=world_size,
    )

    # -- Model -----------------------------------------------------------------
    model = BirdCLEFModel(cfg, is_train=True).to(device)
    # DDP with NCCL backend. find_unused_parameters=False skips the
    # autograd graph scan for unused params, saving overhead every backward pass.
    model = DDP(model, device_ids=[rank], find_unused_parameters=False)

    # -- Optimiser, Scheduler & Loss -------------------------------------------
    optimizer = build_optimizer(model.module, cfg)
    scheduler = CosineAnnealingLR(
        optimizer, T_max=cfg['epochs'], eta_min=cfg['lr'] * 0.01
    )
    criterion = MaskedBCEWithLogitsLoss()

    # torch.amp.GradScaler replaces deprecated torch.cuda.amp.GradScaler.
    scaler = torch.amp.GradScaler('cuda', init_scale=2 ** 16)

    # -- Snapshot directory (rank 0 creates it) --------------------------------
    snapshot_dir = cfg['output_dir'] / 'snapshots'
    if rank == 0:
        snapshot_dir.mkdir(exist_ok=True)

    history:  List[Dict] = []
    best_auc: float      = 0.0

    if rank == 0:
        print('\n' + '=' * 70)
        print(
            f'TRAINING  |  Fold {cfg["train_fold"]}  |  '
            f'{cfg["epochs"]} epochs  |  DDP x {world_size} GPUs'
        )
        print('=' * 70)

    # -- Training loop ---------------------------------------------------------
    for epoch in range(1, cfg['epochs'] + 1):
        # DistributedSampler must be re-seeded each epoch so each rank receives
        # a fresh, different permutation of the data on every pass.
        train_sampler.set_epoch(epoch)
        t0 = time.time()

        train_loss = train_one_epoch(
            model, train_loader, optimizer, criterion,
            scaler, device, epoch, cfg,
        )
        val_loss, val_auc = validate(model, val_loader, criterion, device)
        scheduler.step()

        # -- Cross-rank metric aggregation -------------------------------------
        # Each rank computed metrics on its own data shard. Sum across all ranks
        # then divide by world_size for the true global average.
        avg_train = torch.tensor(train_loss, device=device)
        avg_val   = torch.tensor(val_loss,   device=device)
        avg_auc   = torch.tensor(val_auc,    device=device)
        for t in (avg_train, avg_val, avg_auc):
            dist.all_reduce(t, op=dist.ReduceOp.SUM)
        avg_train_loss = (avg_train / world_size).item()
        avg_val_loss   = (avg_val   / world_size).item()
        avg_val_auc    = (avg_auc   / world_size).item()

        # -- Rank-0 logging and checkpointing ----------------------------------
        # Only rank 0 writes to disk and prints, preventing race conditions
        # where both processes attempt to write to the same checkpoint file.
        if rank == 0:
            elapsed = time.time() - t0
            history.append({
                'epoch':      epoch,
                'train_loss': avg_train_loss,
                'val_loss':   avg_val_loss,
                'val_auc':    avg_val_auc,
            })
            print(
                f'Epoch {epoch:03d}/{cfg["epochs"]}  '
                f'train={avg_train_loss:.4f}  val={avg_val_loss:.4f}  '
                f'auc={avg_val_auc:.4f}  [{elapsed:.0f}s]'
            )

            # Snapshot checkpoint: saved as the unwrapped state_dict
            # (model.module) so it is loadable without DDP at inference time.
            if epoch in cfg['snapshot_epochs']:
                snap_path = snapshot_dir / f'snapshot_epoch{epoch:03d}.pt'
                torch.save(model.module.state_dict(), snap_path)
                print(f'  [SNAP] -> {snap_path.name}')

            # Update best-AUC checkpoint whenever validation AUC improves.
            if avg_val_auc > best_auc:
                best_auc = avg_val_auc
                torch.save(
                    model.module.state_dict(),
                    cfg['output_dir'] / 'best_model.pt',
                )

    # -- Post-training: learning curves + CSV (rank 0 only) --------------------
    if rank == 0:
        print(f'\n[DONE] Best val AUC: {best_auc:.4f}')

        hist_df = pd.DataFrame(history)
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        axes[0].plot(hist_df['epoch'], hist_df['train_loss'], label='train')
        axes[0].plot(hist_df['epoch'], hist_df['val_loss'],   label='val')
        axes[0].set_title('Asymmetric Loss')
        axes[0].legend()
        axes[1].plot(hist_df['epoch'], hist_df['val_auc'], color='steelblue')
        axes[1].set_title('Val Macro ROC-AUC')
        plt.tight_layout()
        plt.show()

        hist_df.to_csv(cfg['output_dir'] / 'training_history.csv', index=False)

    ddp_cleanup()

Writing /kaggle/working/worker.py


## 2 - Training Dispatch

This cell orchestrates the full multi-fold training pipeline across **5 folds**, with each fold running through five sequential stages:

### Setup & Validation
- **Metadata loading** - reads `preprocessed_metadata.csv` and `label_map.csv` from `CFG['preprocessed_dir']`. The metadata contains one row per training clip with `mmap_index` (row offset into the `.npy` array) and `label_id` (integer class label). The label map is injected directly into `CFG['label_map']` as a `{primary_label → label_id}` dict.
- **Array resolution** - resolves paths for the focal and background `.npy` waveform arrays. If `birdclef2026_background.npy` is not found on disk, the focal array is used as a fallback with a `[WARN]` notice.
- **Class count validation** - cross-checks `CFG['num_classes']` against the actual number of unique `label_id` values in the metadata. A mismatch would silently produce wrong-shaped logits at inference (the classification head and target matrix share this dimension), so `num_classes` is corrected in-place if needed.

### Per-Fold Stages

**1 · Isolated configuration** - a shallow copy of `CFG` is made for each fold and patched with `train_fold = fold` and a dedicated `output_dir` (`output_dir/fold_{fold}/`). Isolating the output directory prevents worker snapshots from different folds from overwriting each other.

**2 · DDP process spawning** - `mp.start_processes` launches one Python interpreter process per available GPU (`world_size = torch.cuda.device_count()`). Each process receives its `rank` (0 … world_size−1) alongside the shared config, metadata DataFrame, and array paths, then runs `train_worker` independently. `join=True` blocks the cell until all workers exit cleanly.

**3 · Snapshot averaging** - calls the helper `average_snapshots(snapshot_dir, snapshot_epochs, base_model, fallback_path)`, which must be defined in a preceding cell. It loads the periodic weight snapshots written to `fold_{fold}/snapshots/` at the epochs listed in `CFG['snapshot_epochs']` and averages their parameters into a single `BirdCLEFModel` instantiated in inference mode (`is_train=False`). If no snapshots are found it falls back to `best_model.pt`. The result is moved to CPU and set to `eval()` before export.

**4 · ONNX export** - calls two helper functions that must be defined in a preceding cell. `export_to_onnx(model, path, samples_per_clip)` traces the averaged model into a raw FP32 ONNX graph (`fold_{fold}/birdclef2026_fold{fold}_raw.onnx`). `optimize_onnx_graph(input_path, output_path)` applies constant folding and redundant-node elimination, writing the final optimised graph to the **root** `output_dir` as `birdclef2026_fold{fold}.onnx` so all fold models are co-located for easy ensemble loading. After export, `final_onnx_path` holds the path of the most recently exported fold's ONNX file.

**5 · Memory cleanup** - the averaged PyTorch model is explicitly deleted, followed by `gc.collect()` and `torch.cuda.empty_cache()`, to release both CPU RAM and CUDA memory before the next fold's workers are spawned. This is important when running 5 consecutive folds inside a single Kaggle session with a fixed memory budget. As a consequence, `averaged_model` is **not** in scope after the loop; the benchmark cell in Section 3 must reload a model from a saved checkpoint for the PyTorch sanity check.
```

In [4]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def average_snapshots(
    snapshot_dir:    Path,
    snapshot_epochs: List[int],
    base_model:      nn.Module,
    fallback_path:   Optional[Path] = None,
) -> nn.Module:
    """Average K snapshot state_dicts element-wise to produce an ensemble model."""
    available = [
        snapshot_dir / f'snapshot_epoch{e:03d}.pt'
        for e in snapshot_epochs
        if (snapshot_dir / f'snapshot_epoch{e:03d}.pt').exists()
    ]

    if not available:
        fb = fallback_path or (snapshot_dir.parent / 'best_model.pt')
        print(f'[SNAP] No snapshots found -- loading fallback: {fb.name}')
        base_model.load_state_dict(torch.load(fb, map_location='cpu'))
        return base_model.eval()

    print(f'[SNAP] Averaging {len(available)} snapshots:')
    for p in available:
        print(f'       {p.name}')

    avg_state: Optional[Dict] = None

    for path in available:
        state = torch.load(path, map_location='cpu')

        if avg_state is None:
            avg_state = {k: v.float().clone() for k, v in state.items()}
        else:
            for k in avg_state:
                avg_state[k].add_(state[k].float())

    K = float(len(available))
    for k in avg_state:
        avg_state[k].div_(K)

    base_model.load_state_dict(avg_state)
    base_model.is_train = False
    print(f'[SNAP] Done  (K = {int(K)})')
    return base_model.eval()

def export_to_onnx(
    model:            nn.Module,
    out_path:         str,
    samples_per_clip: int,
) -> str:
    """Trace and export the model to a raw FP32 ONNX file."""
    model.cpu().eval()
    dummy_input = torch.zeros(2, samples_per_clip, dtype=torch.float32)

    torch.onnx.export(
        model,
        dummy_input,
        out_path,
        input_names   = ['waveform'],
        output_names  = ['logits'],
        dynamic_axes  = {
            'waveform': {0: 'batch_size'},
            'logits':   {0: 'batch_size'},
        },
        opset_version       = 18,
        do_constant_folding = True,
        verbose             = False,
    )
    return out_path

def optimize_onnx_graph(fp32_raw_path: str, fp32_opt_path: str) -> str:
    """Apply ORT's full graph optimisation pass and save the result."""
    session_opts = ort.SessionOptions()
    session_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED
    session_opts.optimized_model_filepath  = fp32_opt_path

    _ = ort.InferenceSession(
        fp32_raw_path, sess_options=session_opts,
        providers=['CPUExecutionProvider'],
    )
    return fp32_opt_path

In [5]:
sys.path.insert(0, '/kaggle/working')

# Import worker dependencies directly from the written file
from worker import train_worker, BirdCLEFModel

# -- Load metadata ------------------------------------------------------------
preprocessed_dir = CFG['preprocessed_dir']
meta_df          = pd.read_csv(preprocessed_dir / 'preprocessed_metadata.csv')

# Load the label map and inject it into CFG
label_map_df = pd.read_csv(preprocessed_dir / 'label_map.csv')
CFG['label_map'] = dict(zip(label_map_df['primary_label'], label_map_df['label_id']))

focal_npy = str(preprocessed_dir / 'birdclef2026_focal.npy')
bg_npy    = str(preprocessed_dir / 'birdclef2026_background.npy')

if not Path(bg_npy).exists():
    print('[WARN] Background array not found -- using focal array as background.')
    bg_npy = focal_npy

n_actual = meta_df['label_id'].nunique()
if n_actual != CFG['num_classes']:
    print(f'[WARN] Updating num_classes: {CFG["num_classes"]} -> {n_actual}')
    CFG['num_classes'] = n_actual

world_size = torch.cuda.device_count()

# =============================================================================
# MULTI-FOLD ORCHESTRATOR
# =============================================================================
for fold in range(CFG['n_folds']):
    print("\n" + "="*60)
    print(f"STARTING FOLD {fold} OF {CFG['n_folds']}")
    print("="*60)
    
    # 1. Isolate Configuration for this specific fold
    CFG_fold = CFG.copy()
    CFG_fold['train_fold'] = fold
    
    # CRITICAL: Create an isolated workspace for this fold so worker.py
    # does not overwrite or average snapshots from previous folds.
    fold_dir = CFG['output_dir'] / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    CFG_fold['output_dir'] = fold_dir 
    
    # 2. Spawn DDP worker processes
    print(f'[DDP] Spawning {world_size} GPU worker processes ...')
    mp.start_processes(
        train_worker,
        args=(world_size, CFG_fold, meta_df, focal_npy, bg_npy),
        nprocs=world_size,
        join=True,
        start_method='spawn',
    )
    
    # 3. Snapshot Averaging for this fold
    print(f"\n[SNAP] Averaging snapshots for Fold {fold}...")
    snapshot_dir = fold_dir / 'snapshots'
    averaged_model = BirdCLEFModel(CFG_fold, is_train=False)
    averaged_model = average_snapshots(
        snapshot_dir    = snapshot_dir,
        snapshot_epochs = CFG_fold['snapshot_epochs'],
        base_model      = averaged_model,
        fallback_path   = fold_dir / 'best_model.pt',
    )
    averaged_model.cpu().eval()
    
    # 4. ONNX Export Pipeline
    fp32_raw_path = str(fold_dir / f'birdclef2026_fold{fold}_raw.onnx')
    
    # Save the final optimized ONNX to the root working dir so it is easy to find
    final_onnx_path = str(CFG['output_dir'] / f'birdclef2026_fold{fold}.onnx') 
    
    export_to_onnx(averaged_model, fp32_raw_path, SAMPLES_PER_CLIP)
    optimize_onnx_graph(fp32_raw_path, final_onnx_path)
    
    # 5. Memory Cleanup
    # Destroy the PyTorch graph and garbage collect before the next fold starts
    # to prevent CPU RAM / CUDA OOM errors over the 5 loops.
    del averaged_model
    gc.collect()
    torch.cuda.empty_cache()
    
    print(f"Fold {fold} Complete! Model exported to: {final_onnx_path}")

print("\nALL FOLDS SUCCESSFULLY TRAINED AND EXPORTED!")

[WARN] Background array not found -- using focal array as background.
[WARN] Updating num_classes: 234 -> 183

STARTING FOLD 0 OF 5
[DDP] Spawning 2 GPU worker processes ...
[FOLD 0] Train: 14,889  |  Val: 3,723
[DATA] Loaded 245134 pseudo-labeled background targets.

TRAINING  |  Fold 0  |  30 epochs  |  DDP x 2 GPUs
[DATA] Loaded 245134 pseudo-labeled background targets.


Epoch 002 [train]:   0%|          | 0/232 [00:00<?, ?it/s]

Epoch 001/30  train=0.0916  val=0.0342  auc=0.5351  [162s]


Epoch 003 [train]:   0%|          | 0/232 [00:00<?, ?it/s]

Epoch 002/30  train=0.0490  val=0.0330  auc=0.6039  [43s]


...

Epoch 030 [train]:   0%|          | 0/232 [00:00<?, ?it/s]

Epoch 029/30  train=0.0297  val=0.0141  auc=0.9660  [45s]


Epoch 030/30  train=0.0295  val=0.0139  auc=0.9654  [46s]
  [SNAP] -> snapshot_epoch030.pt

[DONE] Best val AUC: 0.9660
Figure(1300x400)

[SNAP] Averaging snapshots for Fold 4...
[SNAP] Averaging 6 snapshots:
       snapshot_epoch020.pt
       snapshot_epoch022.pt
       snapshot_epoch024.pt
       snapshot_epoch026.pt
       snapshot_epoch028.pt
       snapshot_epoch030.pt
[SNAP] Done  (K = 6)


W0318 18:48:35.860000 24 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0318 18:48:35.862000 24 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0318 18:48:35.864000 24 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0318 18:48:35.866000 24 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


Applied 147 of general pattern rewrite rules.
Fold 4 Complete! Model exported to: /kaggle/working/birdclef2026_fold4.onnx

ALL FOLDS SUCCESSFULLY TRAINED AND EXPORTED!
